# NMF on ModulAir 00689

In [1]:
from sklearn.decomposition import NMF
import numpy as np
import pandas as pd

## Cleaning Data

In [2]:
#importing data from Modulair MOD-000689
data = pd.read_csv('MOD-00689.csv').set_index('timestamp')
data.head()

,id,timestamp_local,sn,rh,temp,bin0,bin1,bin2,bin3,bin4,...,no2,o3,pm1_model_id,pm25_model_id,pm10_model_id,co_model_id,no_model_id,no2_model_id,o3_model_id,ws_scalar
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-12-31T23:59:05Z,577612306,2025-12-31T18:59:05Z,MOD-00689,49.6,0.5,6.743,0.692,0.187,0.038,0.031,...,29.938,38.903,14340.0,14341.0,14342.0,14476.0,14501.0,14551.0,14526.0,0.90
2025-12-31T23:58:05Z,577610344,2025-12-31T18:58:05Z,MOD-00689,49.7,0.5,7.506,0.821,0.251,0.027,0.042,...,29.223,39.248,14340.0,14341.0,14342.0,14476.0,14501.0,14551.0,14526.0,0.97
2025-12-31T23:57:05Z,577610343,2025-12-31T18:57:05Z,MOD-00689,49.8,0.4,7.911,0.835,0.231,0.051,0.034,...,29.449,39.248,14340.0,14341.0,14342.0,14476.0,14501.0,14551.0,14526.0,1.61
2025-12-31T23:56:05Z,577610342,2025-12-31T18:56:05Z,MOD-00689,50.0,0.4,7.611,0.901,0.256,0.050,0.030,...,29.677,38.185,14340.0,14341.0,14342.0,14476.0,14501.0,14551.0,14526.0,1.54
2025-12-31T23:55:05Z,577610341,2025-12-31T18:55:05Z,MOD-00689,49.9,0.4,8.115,0.936,0.324,0.078,0.046,...,30.155,38.190,14340.0,14341.0,14342.0,14476.0,14501.0,14551.0,14526.0,1.50


In [3]:
start = data.index.min()
end = data.index.max()

print(start, end)

2025-04-01T00:00:58Z 2025-12-31T23:59:05Z


In [4]:
#only columns of interest
COLS_TO_INCLUDE = ['timestamp_local','co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
data = data[COLS_TO_INCLUDE]

data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
timestamp,,,,,,,,,,,
2025-12-31T23:59:05Z,2025-12-31T18:59:05Z,775.946,2.402,29.938,38.903,6.743,0.692,0.187,0.038,0.031,0.020
2025-12-31T23:58:05Z,2025-12-31T18:58:05Z,768.897,2.401,29.223,39.248,7.506,0.821,0.251,0.027,0.042,0.015
2025-12-31T23:57:05Z,2025-12-31T18:57:05Z,760.097,2.480,29.449,39.248,7.911,0.835,0.231,0.051,0.034,0.015
2025-12-31T23:56:05Z,2025-12-31T18:56:05Z,784.175,2.519,29.677,38.185,7.611,0.901,0.256,0.050,0.030,0.010
2025-12-31T23:55:05Z,2025-12-31T18:55:05Z,774.804,2.519,30.155,38.190,8.115,0.936,0.324,0.078,0.046,0.012


In [5]:
#removing the UTC time
data = data.reset_index(drop = True)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-31T18:59:05Z,775.946,2.402,29.938,38.903,6.743,0.692,0.187,0.038,0.031,0.020
1,2025-12-31T18:58:05Z,768.897,2.401,29.223,39.248,7.506,0.821,0.251,0.027,0.042,0.015
2,2025-12-31T18:57:05Z,760.097,2.480,29.449,39.248,7.911,0.835,0.231,0.051,0.034,0.015
3,2025-12-31T18:56:05Z,784.175,2.519,29.677,38.185,7.611,0.901,0.256,0.050,0.030,0.010
4,2025-12-31T18:55:05Z,774.804,2.519,30.155,38.190,8.115,0.936,0.324,0.078,0.046,0.012


In [6]:
#converting to datetime
data['timestamp_local'] = pd.to_datetime(data['timestamp_local'],
                                       format='%Y-%m-%dT%H:%M:%SZ',
                                       exact=False)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-31 18:59:05,775.946,2.402,29.938,38.903,6.743,0.692,0.187,0.038,0.031,0.020
1,2025-12-31 18:58:05,768.897,2.401,29.223,39.248,7.506,0.821,0.251,0.027,0.042,0.015
2,2025-12-31 18:57:05,760.097,2.480,29.449,39.248,7.911,0.835,0.231,0.051,0.034,0.015
3,2025-12-31 18:56:05,784.175,2.519,29.677,38.185,7.611,0.901,0.256,0.050,0.030,0.010
4,2025-12-31 18:55:05,774.804,2.519,30.155,38.190,8.115,0.936,0.324,0.078,0.046,0.012


In [7]:
#taking hourly average of df. round to floor of the hour
data = data.groupby(data['timestamp_local'].dt.floor('h')).agg(co = ('co','mean'),
                                                         no2 = ('no2','mean'),
                                                         o3 = ('o3','mean'),
                                                         no = ('no','mean'),
                                                         bin0 = ('bin0','mean'),
                                                         bin1 = ('bin1','mean'),
                                                         bin2 = ('bin2','mean'),
                                                         bin3 = ('bin3','mean'),
                                                         bin4 = ('bin4','mean'),
                                                         bin5 = ('bin5','mean')).reset_index()

data = data.round(decimals = 2)
data = data.dropna()
data

,timestamp_local,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-03-31 20:00:00,834.61,28.18,42.49,1.83,20.77,2.29,1.14,0.44,0.63,0.50
1,2025-03-31 21:00:00,901.42,32.65,44.94,2.30,27.62,3.76,1.45,0.48,0.67,0.55
2,2025-03-31 22:00:00,847.25,25.19,52.69,2.45,23.72,4.49,1.47,0.40,0.48,0.36
3,2025-03-31 23:00:00,740.77,14.71,60.56,2.43,18.32,4.91,1.49,0.33,0.33,0.20
4,2025-04-01 00:00:00,687.76,13.21,63.82,2.30,17.16,3.37,0.98,0.22,0.23,0.15
...,...,...,...,...,...,...,...,...,...,...,...
6471,2025-12-31 14:00:00,701.00,28.14,46.26,2.02,7.36,0.59,0.18,0.03,0.03,0.01
6472,2025-12-31 15:00:00,725.34,28.23,45.66,1.94,7.07,0.60,0.18,0.03,0.03,0.01
6473,2025-12-31 16:00:00,732.31,28.74,43.74,2.43,7.85,0.69,0.21,0.03,0.03,0.02
6474,2025-12-31 17:00:00,761.58,29.63,40.06,2.38,8.35,0.82,0.25,0.04,0.03,0.02


In [8]:
#setting local time as index
data = data.set_index('timestamp_local')
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-03-31 20:00:00,834.61,28.18,42.49,1.83,20.77,2.29,1.14,0.44,0.63,0.50
2025-03-31 21:00:00,901.42,32.65,44.94,2.30,27.62,3.76,1.45,0.48,0.67,0.55
2025-03-31 22:00:00,847.25,25.19,52.69,2.45,23.72,4.49,1.47,0.40,0.48,0.36
2025-03-31 23:00:00,740.77,14.71,60.56,2.43,18.32,4.91,1.49,0.33,0.33,0.20
2025-04-01 00:00:00,687.76,13.21,63.82,2.30,17.16,3.37,0.98,0.22,0.23,0.15


In [9]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()

In [10]:
#scaling
def maximum_absolute_scaling(df):
    # copy the dataframe
    df_scaled = df.copy()
    # apply maximum absolute scaling
    for column in df_scaled.columns:
        df_scaled[column] = df_scaled[column]  / df_scaled[column].abs().max()
    return df_scaled

# call the maximum_absolute_scaling function
data = maximum_absolute_scaling(data)

In [11]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-03-31 20:00:00,0.103370,0.671112,0.502305,0.007002,0.288392,0.090945,0.089623,0.126074,0.188623,0.128866
2025-03-31 21:00:00,0.111645,0.777566,0.531268,0.008801,0.383505,0.149325,0.113994,0.137536,0.200599,0.141753
2025-03-31 22:00:00,0.104935,0.599905,0.622887,0.009375,0.329353,0.178316,0.115566,0.114613,0.143713,0.092784
2025-03-31 23:00:00,0.091747,0.350322,0.715924,0.009298,0.254374,0.194996,0.117138,0.094556,0.098802,0.051546
2025-04-01 00:00:00,0.085182,0.314599,0.754463,0.008801,0.238267,0.133836,0.077044,0.063037,0.068862,0.038660


In [12]:
data.to_csv(r'MOD-000689_timeseries_hourly_scaled.csv')

In [13]:
len(data)

6376

## Implementing NMF

In [14]:
#setting up the NMF
nmf = NMF(n_components = 4, max_iter = 8000)

In [15]:
W = nmf.fit_transform(data)
H = nmf.components_
V = pd.DataFrame(np.dot(W,H), columns=data.columns)
V.index = data.index
V

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-03-31 20:00:00,0.114025,0.670624,0.506164,0.009847,0.257238,0.178728,0.122130,0.108397,0.116166,0.065196
2025-03-31 21:00:00,0.129800,0.776559,0.534072,0.010719,0.355081,0.226573,0.147237,0.126390,0.133818,0.074590
2025-03-31 22:00:00,0.125137,0.598636,0.622327,0.010873,0.320183,0.199614,0.129577,0.112811,0.121789,0.068788
2025-03-31 23:00:00,0.114869,0.348755,0.712373,0.010681,0.261437,0.169311,0.113308,0.102456,0.113626,0.065159
2025-04-01 00:00:00,0.113982,0.312709,0.750921,0.010941,0.240566,0.122092,0.074325,0.065887,0.076352,0.045365
...,...,...,...,...,...,...,...,...,...,...
2025-12-31 14:00:00,0.105432,0.668963,0.544575,0.010302,0.102573,0.020648,0.008418,0.007491,0.013482,0.010539
2025-12-31 15:00:00,0.104702,0.671341,0.537838,0.010233,0.099002,0.020183,0.008647,0.007887,0.013840,0.010699
2025-12-31 16:00:00,0.103769,0.683616,0.515414,0.010014,0.109643,0.024748,0.010488,0.008617,0.014088,0.010651


In [16]:
W_df = pd.DataFrame(W, columns =[f'Feature {i+1}' for i in range(0,4)]) #array-like of shape (n_samples, n_components)
W_df

,Feature 1,Feature 2,Feature 3,Feature 4
0,0.042232,0.109377,0.053291,0.069352
1,0.040019,0.127922,0.075923,0.081263
2,0.051615,0.095369,0.066956,0.071541
3,0.064680,0.050094,0.052471,0.064117
4,0.070070,0.043248,0.047123,0.038850
...,...,...,...,...
6371,0.054933,0.107912,0.016119,0.000089
6372,0.054384,0.108409,0.015349,0.000410
6373,0.051326,0.110986,0.018123,0.001059
6374,0.046195,0.115491,0.020330,0.002602


In [17]:
COLS_TO_INCLUDE.pop(0)
COLS_TO_INCLUDE

['co', 'no', 'no2', 'o3', 'bin0', 'bin1', 'bin2', 'bin3', 'bin4', 'bin5']

In [18]:
H_df = pd.DataFrame(H, index = [f'Feature {i+1}' for i in range(0,4)], columns = COLS_TO_INCLUDE) #array-like of shape (n_components, n_features)
H_df

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Feature 1,1.051950,0.867814,9.277077,0.119915,0.490004,0.000000,0.016784,0.105067,0.242966,0.190524
Feature 2,0.372330,5.753422,0.004871,0.030828,0.054902,0.000000,0.020653,0.010589,0.000000,0.000000
Feature 3,0.462868,0.026345,2.136165,0.024053,4.326047,1.272140,0.318643,0.027724,0.000000,0.000000
Feature 4,0.060670,0.047280,0.000000,0.001854,0.000000,1.599587,1.473366,1.461006,1.527067,0.824058


In [19]:
#converting the results to a dataframe
results = pd.DataFrame(W,index=data.index) #H is time series data of the factors, W is the comp (coeff matrix)
results.columns = ["Factor {}".format(i+1) for i in range(H.T.shape[1])]
results

,Factor 1,Factor 2,Factor 3,Factor 4
timestamp_local,,,,
2025-03-31 20:00:00,0.042232,0.109377,0.053291,0.069352
2025-03-31 21:00:00,0.040019,0.127922,0.075923,0.081263
2025-03-31 22:00:00,0.051615,0.095369,0.066956,0.071541
2025-03-31 23:00:00,0.064680,0.050094,0.052471,0.064117
2025-04-01 00:00:00,0.070070,0.043248,0.047123,0.038850
...,...,...,...,...
2025-12-31 14:00:00,0.054933,0.107912,0.016119,0.000089
2025-12-31 15:00:00,0.054384,0.108409,0.015349,0.000410
2025-12-31 16:00:00,0.051326,0.110986,0.018123,0.001059


In [20]:
COLS_TO_INCLUDE = ['co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
comp = pd.DataFrame(H, index = results.columns, columns = COLS_TO_INCLUDE)
comp

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Factor 1,1.051950,0.867814,9.277077,0.119915,0.490004,0.000000,0.016784,0.105067,0.242966,0.190524
Factor 2,0.372330,5.753422,0.004871,0.030828,0.054902,0.000000,0.020653,0.010589,0.000000,0.000000
Factor 3,0.462868,0.026345,2.136165,0.024053,4.326047,1.272140,0.318643,0.027724,0.000000,0.000000
Factor 4,0.060670,0.047280,0.000000,0.001854,0.000000,1.599587,1.473366,1.461006,1.527067,0.824058


In [21]:
res = []

for col in comp.columns:
    #for each factor, compute its contribution to the species in V
    by_factor = pd.Series(dtype=float)
    for i, factor in enumerate(results.columns):
        contrib = H[i, data.columns.get_loc(col)] * W[:, i].sum()
        by_factor.at[factor] = contrib

    #normalizing by the total amount of that species in the original data
    by_factor /= data[col].sum()

    #adding as a row to the resulting dataframe
    res.append(pd.DataFrame(by_factor).T.rename(index={0: col}))

res = pd.concat(res)
res.columns = results.columns
res['Residual'] = 1 - res.sum(axis=1)

res

,Factor 1,Factor 2,Factor 3,Factor 4,Residual
co,0.584572,0.253683,0.135708,0.007308,0.018728
no,0.692095,0.218151,0.073244,0.002320,0.014190
no2,0.109242,0.887996,0.001750,0.001290,-0.000279
o3,0.891506,0.000574,0.108307,0.000000,-0.000387
bin0,0.172676,0.023722,0.804322,0.000000,-0.000720
bin1,0.000000,0.000000,0.698886,0.361037,-0.059923
bin2,0.031610,0.047689,0.316620,0.601474,0.002607
bin3,0.233611,0.028868,0.032523,0.704134,0.000864
bin4,0.427359,0.000000,0.000000,0.582216,-0.009575
bin5,0.522128,0.000000,0.000000,0.489512,-0.011640


In [22]:
results.to_csv(r'4_factor_results.csv')
comp.to_csv(r'4_factor_comp.csv')
res.to_csv(r'4_factor_resid.csv')